In [7]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

In [8]:
CLEAN_ROOT = Path('../../data/cmhc-rental-clean')
OUT_DIR    = CLEAN_ROOT / 'vis-exports'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CITY_DIR = CLEAN_ROOT / 'on-csd-primary/primary-summary'

# ── CPI, all-items, 2002=100 (Statistics Canada) ─────────────────────────
CPI_YEARS  = list(range(1990, 2026))
CPI_VALUES = [
     78.7,  82.4,  83.2,  84.7,  84.7,  86.8,  88.2,  89.8,  90.6,  92.4,
     95.1,  98.0, 100.0, 102.7, 104.6, 106.9, 108.8, 110.8, 113.3, 113.7,
    116.5, 120.1, 121.8, 123.0, 125.9, 127.4, 129.7, 131.9, 135.0, 137.5,
    138.4, 143.2, 152.9, 158.7, 162.5, 165.6,
]

# ── Ontario rent-control guideline (annual max % increase) ───────────────
RENT_CONTROL = {
    1991: 5.4, 1992: 6.0, 1993: 4.9, 1994: 3.2, 1995: 2.9,
    1996: 2.8, 1997: 2.8, 1998: 3.0, 1999: 3.0, 2000: 2.6,
    2001: 2.9, 2002: 3.9, 2003: 2.9, 2004: 2.9, 2005: 1.5,
    2006: 2.1, 2007: 2.6, 2008: 1.4, 2009: 1.8, 2010: 2.1,
    2011: 0.7, 2012: 3.1, 2013: 2.5, 2014: 0.8, 2015: 1.6,
    2016: 2.0, 2017: 1.5, 2018: 1.8, 2019: 1.8, 2020: 2.2,
    2021: 0.0, 2022: 1.2, 2023: 2.5, 2024: 2.5, 2025: 2.5,
}

# ── Ontario population, Q1 estimates (Statistics Canada) ─────────────────
POP_Q1 = {
    1990: 10_189_985, 1991: 10_355_101, 1992: 10_488_022, 1993: 10_629_994,
    1994: 10_744_762, 1995: 10_875_308, 1996: 11_009_307, 1997: 11_146_270,
    1998: 11_292_059, 1999: 11_419_589, 2000: 11_576_994, 2001: 11_771_945,
    2002: 11_979_906, 2003: 12_155_691, 2004: 12_303_516, 2005: 12_444_755,
    2006: 12_587_149, 2007: 12_703_327, 2008: 12_814_686, 2009: 12_932_742,
    2010: 13_059_426, 2011: 13_199_081, 2012: 13_325_337, 2013: 13_446_276,
    2014: 13_563_311, 2015: 13_657_423, 2016: 13_774_364, 2017: 13_975_516,
    2018: 14_199_811, 2019: 14_449_986, 2020: 14_718_155, 2021: 14_761_811,
    2022: 14_947_417, 2023: 15_305_369, 2024: 15_823_956, 2025: 16_191_372,
}

# ── Ontario median total income (Statistics Canada, ends 2024) ────────────
INCOME_YEARS   = list(range(1990, 2025))
INCOME_ONTARIO = [
    39955, 38335, 39219, 38623, 38675, 39634, 39731, 40451, 42317, 45205,
    47696, 49271, 49969, 51950, 52590, 54864, 55706, 56866, 59265, 57655,
    58716, 59791, 61162, 60402, 64693, 64915, 66007, 69400, 73357, 73023,
    78782, 81954, 83272, 88384, 90200,
]

BASE_YEAR  = 1990
INDEX_BASE = 1000


def save(df, stem):
    """Write df to OUT_DIR as both CSV and JSON (list of row dicts)."""
    csv_path  = OUT_DIR / f'{stem}.csv'
    json_path = OUT_DIR / f'{stem}.json'
    df.to_csv(csv_path, index=False)
    records = df.where(df.notna(), None).to_dict(orient='records')
    with open(json_path, 'w') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f'  saved: {csv_path.name}  ({len(df)} rows x {len(df.columns)} cols)')
    print(f'  saved: {json_path.name}')

## VIS 1 — Benchmark Index Series (1990–2025)

Line chart mirroring Section 2A of `explore_rental_data.ipynb`: Ontario average rent,
CPI, the compounded rent-control guideline, and median income — all re-indexed to
1000 in 1990 so growth rates are directly comparable across series.

`income_index` is null for 2025 (Statistics Canada has not yet published that year).

In [9]:
ontario = pd.read_csv(
    CLEAN_ROOT / 'can-pr-primary/primary-summary/ontario.csv'
).set_index('Year')

# Rent: ratio to base year
rent_idx = INDEX_BASE * ontario['Average Rent'] / ontario.loc[BASE_YEAR, 'Average Rent']

# CPI: ratio to base year
cpi_s   = pd.Series(CPI_VALUES, index=CPI_YEARS)
cpi_idx = INDEX_BASE * cpi_s / cpi_s[BASE_YEAR]

# Rent control: annual compound from 1990 base
rc_idx = pd.Series(index=range(BASE_YEAR, 2026), dtype=float)
rc_idx[BASE_YEAR] = INDEX_BASE
for yr in range(BASE_YEAR + 1, 2026):
    rc_idx[yr] = rc_idx[yr - 1] * (1 + RENT_CONTROL.get(yr, 0.0) / 100)

# Income: ratio to base year (ends 2024 — 2025 stays NaN)
income_s   = pd.Series(INCOME_ONTARIO, index=INCOME_YEARS)
income_idx = INDEX_BASE * income_s / income_s[BASE_YEAR]

vis1 = pd.DataFrame({
    'rent_index':         rent_idx,
    'cpi_index':          cpi_idx,
    'rent_control_index': rc_idx,
    'income_index':       income_idx,
}).loc[1990:2025].round(2).reset_index().rename(columns={'Year': 'year'})

save(vis1, 'vis1_benchmark_index')
vis1.tail(4)

  saved: vis1_benchmark_index.csv  (36 rows x 5 cols)
  saved: vis1_benchmark_index.json


,index,rent_index,cpi_index,rent_control_index,income_index
32,2022,2559.03,1942.82,2196.33,2084.14
33,2023,2793.40,2016.52,2251.23,2212.09
34,2024,2892.36,2064.80,2307.51,2257.54
35,2025,3003.47,2104.19,2365.20,NaN


## VIS 2 — City Rent vs. Rent-Control Guideline (2018–2025)

Paired horizontal bar chart: for each Ontario CSD city, actual rent increase from
2018 to 2025 alongside the increase that would have resulted from following the
province-wide guideline every year.

- `actual_rent_increase` / `guideline_rent_increase`: absolute dollar increase ($/month)
- `actual_pct_increase` / `guideline_pct_increase`: percentage increase

The guideline percentage is the same for every city (compounding 2019–2025 rates);
the dollar amounts differ because each city starts from a different 2018 rent level.
Cities with missing 2018 or 2025 data are omitted.

In [10]:
# Compound the guideline rates from 2019 through 2025 (rates applied on top of 2018 base)
guideline_factor = 1.0
for yr in range(2019, 2026):
    guideline_factor *= (1 + RENT_CONTROL[yr] / 100)
guideline_pct = round((guideline_factor - 1) * 100, 2)
print(f'Province-wide guideline compound increase 2018→2025: {guideline_pct:.2f}%')

rows = []
for fp in tqdm(sorted(CITY_DIR.glob('*.csv')), desc='cities'):
    df = pd.read_csv(fp).set_index('Year')
    try:
        r_2018 = df.loc[2018, 'Average Rent']
        r_2025 = df.loc[2025, 'Average Rent']
    except KeyError:
        continue
    if pd.isna(r_2018) or pd.isna(r_2025):
        continue
    actual_pct = round((r_2025 / r_2018 - 1) * 100, 2)
    rows.append({
        'city':                    fp.stem,
        'actual_rent_increase':    round(r_2025 - r_2018, 2),
        'guideline_rent_increase': round(r_2018 * (guideline_factor - 1), 2),
        'actual_pct_increase':     actual_pct,
        'excess_pct_increase':     round(actual_pct - guideline_pct, 2),
    })

vis2 = pd.DataFrame(rows).sort_values('actual_pct_increase', ascending=False).reset_index(drop=True)
save(vis2, 'vis2_city_rent_vs_guideline')
vis2.head()

Province-wide guideline compound increase 2018→2025: 13.38%


cities:   0%|          | 0/34 [00:00<?, ?it/s]

  saved: vis2_city_rent_vs_guideline.csv  (32 rows x 5 cols)
  saved: vis2_city_rent_vs_guideline.json


,city,actual_rent_increase,guideline_rent_increase,actual_pct_increase,excess_pct_increase
0,newmarket,923.0,161.27,76.60,63.22
1,cambridge,771.0,145.35,70.99,57.61
2,chatham-kent,533.0,101.85,70.04,56.66
3,niagara falls,551.0,128.75,57.28,43.90
4,guelph,626.0,151.64,55.25,41.87


## VIS 3 — Supply vs. Rent Growth (Ontario Year-over-Year)

Two scatter plots mirroring Sections 3A and 3B of `explore_rental_data.ipynb`.

**VIS 3A** — raw % change in primary rental units (x) vs % change in average rent (y),
each year 1991–2025.

**VIS 3B** — demand-adjusted supply: absolute year-over-year change in rental units
per 1,000 Ontario residents (x) vs % change in average rent (y). Using the absolute
delta in density (rather than a % change) keeps the units interpretable as
units-per-1k-people added or lost each year.

In [11]:
# VIS 3A: % change in units and % change in rent, Ontario level
vis3a = pd.DataFrame({
    'pct_units': ontario['Units'].pct_change() * 100,
    'pct_rent':  ontario['Average Rent'].pct_change() * 100,
}).dropna().round(3).reset_index().rename(columns={'Year': 'year'})

save(vis3a, 'vis3a_supply_vs_rent_yoy')
print()

# VIS 3B: absolute YoY change in units per 1,000 residents, and % change in rent
pop_s        = pd.Series(POP_Q1).sort_index()
units_per_1k = ontario['Units'] / (pop_s / 1_000)

vis3b = pd.DataFrame({
    'delta_units_per_1k': units_per_1k.diff(),
    'pct_rent':           ontario['Average Rent'].pct_change() * 100,
}).dropna().round(4).reset_index().rename(columns={'Year': 'year'})

save(vis3b, 'vis3b_demand_adjusted_supply_vs_rent')
print()
print(f'vis3a: {len(vis3a)} rows  |  vis3b: {len(vis3b)} rows')

  saved: vis3a_supply_vs_rent_yoy.csv  (35 rows x 3 cols)
  saved: vis3a_supply_vs_rent_yoy.json

  saved: vis3b_demand_adjusted_supply_vs_rent.csv  (35 rows x 3 cols)
  saved: vis3b_demand_adjusted_supply_vs_rent.json

vis3a: 35 rows  |  vis3b: 35 rows


## VIS 4 — City Supply vs. Rent Growth by Period

Scatter mirroring Section 4 of `explore_rental_data.ipynb`, restricted to the three
most recent seven-year periods (2004–2011, 2011–2018, 2018–2025). Each row is a city;
columns give the percentage change in primary rental units and average rent over each
period endpoint pair.

Only cities with complete data at all three period endpoint pairs are included so
that the same set of cities appears across all three panels.

In [12]:
VIS4_PERIODS = [(2004, 2011), (2011, 2018), (2018, 2025)]


def period_pcts(start_yr, end_yr):
    """Return Series of {city: {pct_units_S_E, pct_rent_S_E}} for cities with full data."""
    rows = {}
    for fp in sorted(CITY_DIR.glob('*.csv')):
        df = pd.read_csv(fp).set_index('Year')
        try:
            r0, r1 = df.loc[start_yr, 'Average Rent'], df.loc[end_yr, 'Average Rent']
            u0, u1 = df.loc[start_yr, 'Units'],        df.loc[end_yr, 'Units']
        except KeyError:
            continue
        if any(pd.isna(v) for v in (r0, r1, u0, u1)) or u0 == 0:
            continue
        rows[fp.stem] = {
            f'pct_units_{start_yr}_{end_yr}': round((u1 - u0) / u0 * 100, 2),
            f'pct_rent_{start_yr}_{end_yr}':  round((r1 - r0) / r0 * 100, 2),
        }
    return pd.DataFrame(rows).T


period_dfs = [period_pcts(s, e) for s, e in tqdm(VIS4_PERIODS, desc='periods')]

# Inner join: keep only cities present in all three periods
vis4 = period_dfs[0].join(period_dfs[1], how='inner').join(period_dfs[2], how='inner')
vis4 = vis4.reset_index().rename(columns={'index': 'city'})

save(vis4, 'vis4_city_period_supply_rent')
print(f'  {len(vis4)} cities with complete data across all three periods')
vis4.head()

periods:   0%|          | 0/3 [00:00<?, ?it/s]

  saved: vis4_city_period_supply_rent.csv  (32 rows x 7 cols)
  saved: vis4_city_period_supply_rent.json
  32 cities with complete data across all three periods


,city,pct_units_2004_2011,pct_rent_2004_2011,pct_units_2011_2018,pct_rent_2011_2018,pct_units_2018_2025,pct_rent_2018_2025
0,ajax,0.88,1.17,20.03,20.69,32.52,38.76
1,barrie,-2.64,9.94,11.21,32.37,6.55,31.91
2,brampton,5.49,7.09,4.88,22.31,8.35,49.54
3,brantford,-2.16,14.80,4.56,25.91,4.60,53.98
4,burlington,5.38,15.99,5.80,28.12,0.80,41.82
